# **Mapping Topik BERTopic -> 15 Aspek (rujuk FiQA)**

Memetakan 46 topik BERTopic ke **maksimal 15 aspek**, dikelompokkan di bawah 4 kategori FiQA: `Corporate`, `Stock`, `Market`, `Economy` (Maia et al. 2018; Lengkeek et al. 2023). Topik semakna digabung sehingga jumlah aspek ringkas.

- Input: `data/processed/sentences_with_topic.csv` + `data/processed/topic_info.csv`.
- Output: `data/processed/mapping_aspek.csv` (tabel cek per topik) + `data/processed/sentences_with_aspect.csv` (kalimat + aspek).

Mapping ini **usulan awal berbasis keyword** - keputusan final tetap manual. Topik `-1` (outlier) & topik tak jelas -> **Lainnya** (bukan salah satu dari 15 aspek).

In [ ]:
from pathlib import Path
import pandas as pd

sent = pd.read_csv("data/processed/sentences_with_topic.csv", dtype={"id_artikel": str})
info = pd.read_csv("data/processed/topic_info.csv")
print(f"Kalimat: {len(sent):,} | Topik: {len(info)}")

## Perbaikan: isi 3 kalimat representatif untuk topik -1

Di `topic_info.csv`, topik `-1` (outlier) dibuat tanpa kalimat representatif. Agar bisa dicek manual, ambil 3 contoh kalimat `-1` langsung dari data (tidak perlu re-run BERTopic).

In [ ]:
rep_cols = ["Representative_1", "Representative_2", "Representative_3"]
mask_min1 = info["Topic"] == -1
if mask_min1.any():
    contoh = sent.loc[sent["topic"] == -1, "kalimat"].head(3).tolist()
    contoh = (contoh + ["", "", ""])[:3]
    for col, val in zip(rep_cols, contoh):
        info.loc[mask_min1, col] = val
    print("Contoh kalimat topik -1 terisi:")
    for c in contoh:
        print("  -", c[:100])

## 15 aspek final + pemetaan tiap topik

`ASPEK` mendefinisikan 15 aspek (nama + kategori FiQA). `TOPIC2ASPEK` memetakan tiap topik ke salah satu aspek, atau ke `Lainnya`. Edit bebas setelah membaca kalimat representatif.

In [ ]:
# 15 ASPEK FINAL: nama -> kategori FiQA
ASPEK = {
    # CORPORATE (6)
    "Kinerja Keuangan & Laba":            "Corporate",
    "Kredit & Kualitas Aset":             "Corporate",
    "Dividen & Aksi Korporasi":           "Corporate",
    "Manajemen & Kepemimpinan":           "Corporate",
    "Layanan & Produk":                   "Corporate",
    "Strategi & Ekspansi Bisnis":         "Corporate",
    # STOCK (4)
    "Pergerakan Saham Bank":              "Stock",
    "Aksi Investor Asing":                "Stock",
    "Saham Spesifik Non-Bank":            "Stock",
    "Valuasi & Kapitalisasi":             "Stock",
    # MARKET (3)
    "Pergerakan IHSG & Indeks":           "Market",
    "Sektor & Transaksi Pasar":           "Market",
    "Bursa Global":                       "Market",
    # ECONOMY (2)
    "Kebijakan Moneter & Nilai Tukar":    "Economy",
    "Makroekonomi & Faktor Global":       "Economy",
}
assert len(ASPEK) == 15, f"harus 15 aspek, sekarang {len(ASPEK)}"

# pemetaan tiap topik -> nama aspek (atau 'Lainnya')
TOPIC2ASPEK = {
    -1: "Lainnya", 11: "Lainnya", 17: "Lainnya", 39: "Lainnya",
    # Corporate
    14: "Kinerja Keuangan & Laba",
    13: "Kredit & Kualitas Aset", 23: "Kredit & Kualitas Aset", 34: "Kredit & Kualitas Aset", 36: "Kredit & Kualitas Aset",
    7:  "Dividen & Aksi Korporasi", 21: "Dividen & Aksi Korporasi", 26: "Dividen & Aksi Korporasi",
    16: "Manajemen & Kepemimpinan",
    10: "Layanan & Produk", 22: "Layanan & Produk",
    2:  "Strategi & Ekspansi Bisnis", 9: "Strategi & Ekspansi Bisnis", 19: "Strategi & Ekspansi Bisnis",
    # Stock
    1:  "Pergerakan Saham Bank", 32: "Pergerakan Saham Bank", 44: "Pergerakan Saham Bank",
    5:  "Aksi Investor Asing", 8: "Aksi Investor Asing", 20: "Aksi Investor Asing", 35: "Aksi Investor Asing",
    31: "Saham Spesifik Non-Bank", 33: "Saham Spesifik Non-Bank", 45: "Saham Spesifik Non-Bank",
    40: "Valuasi & Kapitalisasi",
    # Market
    0:  "Pergerakan IHSG & Indeks", 12: "Pergerakan IHSG & Indeks", 15: "Pergerakan IHSG & Indeks",
    4:  "Sektor & Transaksi Pasar", 6: "Sektor & Transaksi Pasar", 37: "Sektor & Transaksi Pasar",
    25: "Bursa Global", 38: "Bursa Global", 41: "Bursa Global",
    # Economy
    3:  "Kebijakan Moneter & Nilai Tukar", 18: "Kebijakan Moneter & Nilai Tukar", 24: "Kebijakan Moneter & Nilai Tukar",
    27: "Kebijakan Moneter & Nilai Tukar", 28: "Kebijakan Moneter & Nilai Tukar",
    29: "Makroekonomi & Faktor Global", 30: "Makroekonomi & Faktor Global", 42: "Makroekonomi & Faktor Global", 43: "Makroekonomi & Faktor Global",
}

# cek: semua topik tercakup
topik_data = set(int(t) for t in info["Topic"])
belum = sorted(topik_data - set(TOPIC2ASPEK))
print("Topik belum dipetakan:", belum if belum else "tidak ada (semua tercakup)")
print(f"Jumlah aspek: {len(ASPEK)} (+ Lainnya)")

## Tabel mapping per topik (untuk dicek manual)

In [ ]:
def aspek_of(t):
    return TOPIC2ASPEK.get(int(t), "Lainnya")
def kategori_of(a):
    return ASPEK.get(a, "Lainnya")

info["nama_aspek"]   = info["Topic"].map(aspek_of)
info["kategori_fiqa"] = info["nama_aspek"].map(kategori_of)

mapping = info[["Topic", "Count", "kategori_fiqa", "nama_aspek", "Keywords",
                "Representative_1", "Representative_2", "Representative_3"]].copy()
mapping = mapping.sort_values(["kategori_fiqa", "nama_aspek", "Topic"]).reset_index(drop=True)
mapping

## Ringkasan: distribusi per aspek & per kategori

In [ ]:
tot = info["Count"].sum()
print("Distribusi kalimat per ASPEK:")
for a, v in info.groupby("nama_aspek")["Count"].sum().sort_values(ascending=False).items():
    print(f"  {a:34} {v:>6,} ({v/tot*100:4.1f}%)")
print("\nDistribusi kalimat per KATEGORI FiQA:")
for k, v in info.groupby("kategori_fiqa")["Count"].sum().sort_values(ascending=False).items():
    print(f"  {k:12} {v:>6,} ({v/tot*100:4.1f}%)")

## Terapkan ke tiap kalimat & simpan 2 output

In [ ]:
sent["nama_aspek"]   = sent["topic"].map(aspek_of)
sent["kategori_fiqa"] = sent["nama_aspek"].map(kategori_of)

OUT = Path("data/processed")
mapping.to_csv(OUT / "mapping_aspek.csv", index=False, encoding="utf-8-sig")
sent.to_csv(OUT / "sentences_with_aspect.csv", index=False, encoding="utf-8-sig")
print(f"[1] mapping_aspek.csv -> {len(mapping)} topik, {info['nama_aspek'].nunique()-1} aspek + Lainnya")
print(f"[2] sentences_with_aspect.csv -> {len(sent):,} kalimat")

## Langkah manual berikutnya

1. Buka `mapping_aspek.csv`, baca `Keywords` + 3 kalimat representatif, **koreksi** penempatan topik ke aspek bila perlu.
2. Tulis **definisi 1 kalimat + contoh batas** tiap aspek di `daftar_aspek.md`.
3. **Validasi penetapan aspek pada sampel**, lalu **kunci** daftar aspek sebelum anotasi sentimen (Fase 04).